# Basics of ITensors

As far as we are concerned, a tensor is a multi-dimensional table. Each element a rank-$N$ tensor is accessed by providing an assignment to the $N$ (integer) indices.

In this course we will use the `ITensors` library (Julia fashion) to work with tensors. The documentation of the library can be found [here](https://docs.itensor.org/ITensors/stable/). In this notebook we will explore some basics of `ITensors`. 

In [3]:
using Pkg
Pkg.activate("..")
using LinearAlgebra
using ITensors

  Activating project at `~/Work/Workspace/Unimi/Corsi/TN_Ulm/Materiale_dispense/Notebooks`


## Indices

Tensors as multi-indexed objects. The most fundamental entity we have to introduce is thus the `Index`.

In [5]:
#Index is a type that represents the indices of tensors in ITensors.jl. It is used to specify the dimensions and properties of the indices when creating tensors. For example, you can create an index with a specific dimension and name like this:

i=Index(2)


(dim=2|id=832)

We see that an index is assigned an `id`. We do not need to remember the `id`, which is something useful to the library. 

Indices can be assigned tags. A tag is a string, and can be used to add useful information.

In [43]:
j=Index(2;tags="TLS")

(dim=2|id=75|"TLS")

You can extract the tags of an index

In [44]:
tags(j)

"TLS"

The tags of an index can be overwritten by

In [45]:
j=settags(j, "BL,") #Mind the comma, it is needed!


(dim=2|id=75|"BL")

An index can have one or more tags, forming the `TagSet`.

In [46]:
j = addtags(j, "TLS,")

(dim=2|id=75|"BL,TLS")

Tags can be used to extract the elements of an array of index that have the tag

In [49]:
appo = [i,j]

2-element Vector{Index{Int64}}:
 (dim=2|id=832)
 (dim=2|id=75|"BL,TLS")

In [50]:
appo[hastags.(appo, "TLS")]

1-element Vector{Index{Int64}}:
 (dim=2|id=75|"BL,TLS")

There are other properties of indices, such as prime level, which we will discuss further on.

## Tensors

Tensors can be constructed starting from a set of indices. Lets take the two indices `i` and `j` defined on the previous part, characterize them both as the indices of a rank-$1$ tensor providing the state of two Spin 1/2 particles.

In [64]:
i=settags(i,"Spin 1/2,")
j=settags(j,"Spin 1/2,")    

(dim=2|id=75|"Spin1/2")

To actually define the tensors we need to construct one by starting from its indices and the type of the values in the tensor.

In [65]:
ψ1 =ITensor(ComplexF64,i)

ITensor ord=1 (dim=2|id=832|"Spin1/2")
NDTensors.EmptyStorage{ComplexF64, NDTensors.Dense{ComplexF64, Vector{ComplexF64}}}

In [67]:
@show ψ1

ψ1 = ITensor ord=1
Dim 1: (dim=2|id=832|"Spin1/2")
NDTensors.EmptyStorage{ComplexF64, NDTensors.Dense{ComplexF64, Vector{ComplexF64}}}
 2-element




ITensor ord=1 (dim=2|id=832|"Spin1/2")
NDTensors.EmptyStorage{ComplexF64, NDTensors.Dense{ComplexF64, Vector{ComplexF64}}}

The tensor $\psi1$ is, so far, empty: the components of the state are yet unspecified. We can assign values componentwise as follows

In [71]:
# state |+> = (|0> + |1>)/sqrt(2)
ψ1[1] = 1/sqrt(2)
ψ1[2] = 1/sqrt(2)
@show ψ1


ψ1 = ITensor ord=1
Dim 1: (dim=2|id=832|"Spin1/2")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 2-element
 0.7071067811865475 + 0.0im
 0.7071067811865475 + 0.0im



ITensor ord=1 (dim=2|id=832|"Spin1/2")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

Alternatively we can set the components direcly by assigning the values in an array

In [74]:
stateMinus = ComplexF64[1/sqrt(2), -1/sqrt(2)]
ψ2 = ITensor(ComplexF64,stateMinus, j)
#or simply
#ψ2 = ITensor(ComplexF64, j)

ITensor ord=1 (dim=2|id=75|"Spin1/2")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

We can use the indices `i` and `j` to build an operator acting on a qubit

In [80]:
opMat3 = ComplexF64.([1. 0.;0. -1.])
pauli3 = ITensor(ComplexF64, opMat3, i, j)

ITensor ord=2 (dim=2|id=832|"Spin1/2") (dim=2|id=75|"Spin1/2")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

In [81]:
@show pauli3

pauli3 = ITensor ord=2
Dim 1: (dim=2|id=832|"Spin1/2")
Dim 2: (dim=2|id=75|"Spin1/2")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 2×2
 1.0 + 0.0im   0.0 + 0.0im
 0.0 + 0.0im  -1.0 + 0.0im



ITensor ord=2 (dim=2|id=832|"Spin1/2") (dim=2|id=75|"Spin1/2")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

Let's see what happens if we contract the operator with the state ψ1:

In [84]:
out  = pauli3 * ψ1
@show out

out = ITensor ord=1
Dim 1: (dim=2|id=75|"Spin1/2")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 2-element
  0.7071067811865475 + 0.0im
 -0.7071067811865475 + 0.0im



ITensor ord=1 (dim=2|id=75|"Spin1/2")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

"It's a kind of magic": the `*` operation as implemented the contraction bewteen the rank-$2$ tensor (operator) `pauli3` and the rank-$1$ tensor $\psi 1$, namely

$
\sum_{i,j=1}^2 M_{i,j}  \psi1_i  \to \phi_j \quad j=1,2
$

It did this because it recognized that there were repeated indices (the `i` index in the state and the same `i` index in the operator) and inferred a contraction over the indices.

As tou can see, the result is a rank-$1$ tensor indexed by `j` (compare the `id`).

Interesting but...wrong...or better: the result is correct: $\sigma_z \ket{+} = \ket{-}$ but this happened by chance...because `pauli3` is symmetric.

We can inspect this issue by using this custom operator

In [90]:
myOp = ITensor(ComplexF64,ComplexF64.([1 2; 3 4]), i, j)

ITensor ord=2 (dim=2|id=832|"Spin1/2") (dim=2|id=75|"Spin1/2")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

In [92]:
@show myOp * ψ1

myOp * ψ1 = ITensor ord=1
Dim 1: (dim=2|id=75|"Spin1/2")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 2-element
  2.82842712474619 + 0.0im
 4.242640687119285 + 0.0im



ITensor ord=1 (dim=2|id=75|"Spin1/2")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

whereas the matrix-vector multiplication we aimed to was:

In [93]:
ComplexF64.([1 2; 3 4]) * [1/sqrt(2), 1/sqrt(2)]

2-element Vector{ComplexF64}:
 2.1213203435596424 + 0.0im
  4.949747468305832 + 0.0im

So: `ITensor` is indeed be able to perform contractions, but we need to be careful. 

Let's take a step back. 

The index `i` is the _physical_ index of a spin 1/2 particle. It indexes the elements of a basis of an Hilbert space isomorphic to $\mathbb{C}^2$.

The index `j` plays the same role but refers (morally) to the Hilbert space of another system. 

In practice, we must see the states $\psi1$ and $\psi2$ defined by means of two different indices, as objects living in two different Hilbert spaces $\mathcal{H}_1$ and $\mathcal{H}_2$

The correct way do define operators onto $\mathcal{H}_1$ that are able to correctly act on states in the same space is:

$O = \sum_{i'=1}^{2} \sum_{i=1}^2 O_{i',i}\ket{i'}\bra{i}$

In [ ]:
pauli3OK = ITensor(ComplexF64, ComplexF64.([1. 0.;0. -1.]), i',i)

ITensor ord=2 (dim=2|id=75|"Spin1/2")' (dim=2|id=832|"Spin1/2")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

In [119]:
inds(pauli3OK)

((dim=2|id=832|"Spin1/2")', (dim=2|id=832|"Spin1/2"))

__Note:__ the use of the index `i` to index the columns and of `i'`to index the rows. The "prime" in `i'` primes the index: the stored index is the same, but it has an additional property , the prime level.
The prime level of an index can be inspected thgrough the `plev` accessor function.

In [99]:
plev.([i,i'])

2-element Vector{Int64}:
 0
 1

The application of the operator `pauli3OK` to the state $\psi1$ returns 

The proper way to define an operator on the Hilbert space of the a TLS is:

In [104]:
ϕ1  = pauli3OK * ψ1
@show ϕ1

ϕ1 = ITensor ord=1
Dim 1: (dim=2|id=832|"Spin1/2")'
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 2-element
  0.7071067811865475 + 0.0im
 -0.7071067811865475 + 0.0im



ITensor ord=1 (dim=2|id=832|"Spin1/2")'
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

If we want to have the output state with the same (unprimed) index `i` we need to tell `ITensors`

In [105]:
ϕ1  = noprime(pauli3OK * ψ1)
@show ϕ1

ϕ1 = ITensor ord=1
Dim 1: (dim=2|id=832|"Spin1/2")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 2-element
  0.7071067811865475 + 0.0im
 -0.7071067811865475 + 0.0im



ITensor ord=1 (dim=2|id=832|"Spin1/2")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

At last: if we want to extract the "table" of the tensor we can use

In [107]:
matrix(pauli3OK, i', i)

2×2 Matrix{ComplexF64}:
 1.0+0.0im   0.0+0.0im
 0.0+0.0im  -1.0+0.0im

In [110]:
@show matrix(myOp, i, j)
@show matrix(myOp, j, i)

matrix(myOp, i, j) = ComplexF64[1.0 + 0.0im 2.0 + 0.0im; 3.0 + 0.0im 4.0 + 0.0im]
matrix(myOp, j, i) = ComplexF64[1.0 + 0.0im 3.0 + 0.0im; 2.0 + 0.0im 4.0 + 0.0im]


2×2 Matrix{ComplexF64}:
 1.0+0.0im  3.0+0.0im
 2.0+0.0im  4.0+0.0im

### Exercise

For the sake of completeness, let us define the Pauli matrices $\sigma_x,\sigma_y$ and $\sigma_z$ onto the spaces $\mathcal{H}_1$ and $\mathcal{H}_2$.

In [127]:
pauliMatX = ComplexF64.([0. 1.;1. 0.])
pauliMatY = ComplexF64.([0. -im; im 0.])
pauliMatZ = ComplexF64.([1. 0.;0. -1.])
pauliMatId = ComplexF64.([1. 0.;0. 1.])

2×2 Matrix{ComplexF64}:
 1.0+0.0im  0.0+0.0im
 0.0+0.0im  1.0+0.0im

In [156]:
pauliMats = [pauliMatX, pauliMatY, pauliMatZ, pauliMatId]
# In H_1 (index i)
pauliOps1 = [ITensor(ComplexF64, pauliMat, i', i) for pauliMat in pauliMats]
# In H_2 (index j)
pauliOps2 = [ITensor(ComplexF64, pauliMat, j', j) for pauliMat in pauliMats]

#tuple(array...) converts an array into a tuple, which allows us to unpack the elements of the array into separate variables. In this case, we are unpacking the 4 Pauli operators for each index into separate variables for easier access and readability.

σx1,σy1,σz1,id1 = tuple(pauliOps1...)
σx2,σy2,σz2,id2 = tuple(pauliOps2...)



(ITensor ord=2
Dim 1: (dim=2|id=75|"Spin1/2")'
Dim 2: (dim=2|id=75|"Spin1/2")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 2×2
 0.0 + 0.0im  1.0 + 0.0im
 1.0 + 0.0im  0.0 + 0.0im
, ITensor ord=2
Dim 1: (dim=2|id=75|"Spin1/2")'
Dim 2: (dim=2|id=75|"Spin1/2")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 2×2
 0.0 + 0.0im  0.0 - 1.0im
 0.0 + 1.0im  0.0 + 0.0im
, ITensor ord=2
Dim 1: (dim=2|id=75|"Spin1/2")'
Dim 2: (dim=2|id=75|"Spin1/2")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 2×2
 1.0 + 0.0im   0.0 + 0.0im
 0.0 + 0.0im  -1.0 + 0.0im
, ITensor ord=2
Dim 1: (dim=2|id=75|"Spin1/2")'
Dim 2: (dim=2|id=75|"Spin1/2")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 2×2
 1.0 + 0.0im  0.0 + 0.0im
 0.0 + 0.0im  1.0 + 0.0im
)

## Some operations

### Norm

The _norm_ of a state can be computed either as

In [111]:
dag(ψ1) * ψ1

ITensor ord=0
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

We obtain a rank-$0$ tensor, i.e. a scalar. We can extract the value:

In [113]:
scalar(dag(ψ1) * ψ1)

0.9999999999999998 + 0.0im

Even simpler:

In [114]:
norm(ψ1)

0.9999999999999999

### Inner product

In [115]:
η = ITensor(ComplexF64, ComplexF64.([1,0]), i)

ITensor ord=1 (dim=2|id=832|"Spin1/2")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

$\braket{\eta|\psi1}$

In [118]:
scalar(dag(η) * ψ1)

0.7071067811865475 + 0.0im

It should be clear that the order of the arguments does not matter: the contraction occurs over repeated indices

In [123]:
scalar(ψ1 * dag(η))

0.7071067811865475 + 0.0im

### Expectation value

In [137]:
scalar(prime(dag(ψ1)) * σx1 * ψ1)

0.9999999999999998 + 0.0im

or

In [142]:
inner(ψ1', σx1, ψ1)

0.9999999999999998 + 0.0im

### Outer product

If we want to build the density matrix corresponding to the state $\psi1$, i.e.

$ \rho = \ket{\psi1} \bra{\psi1}$

In [124]:
@show ψ1 * prime(dag(ψ1))

ψ1 * prime(dag(ψ1)) = ITensor ord=2
Dim 1: (dim=2|id=832|"Spin1/2")
Dim 2: (dim=2|id=832|"Spin1/2")'
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 2×2
 0.4999999999999999 + 0.0im  0.4999999999999999 + 0.0im
 0.4999999999999999 + 0.0im  0.4999999999999999 + 0.0im



ITensor ord=2 (dim=2|id=832|"Spin1/2") (dim=2|id=832|"Spin1/2")'
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

The `dag` provides a "view" on the state that conjugates the elements, the `prime` avoids unwanted contractions.

### Trace

$ \rho = \ket{\psi1} \bra{\psi1}$

In [207]:
ρ1 = ψ1 * prime(dag(ψ1))
matrix(ρ1, i, i')

2×2 Matrix{ComplexF64}:
 0.5+0.0im  0.5+0.0im
 0.5+0.0im  0.5+0.0im

The trace can be obtained by means of a delta tensor

In [209]:
scalar(ρ1 * delta(i, i'))

0.9999999999999998 + 0.0im

## Composite systems

Let's introduce yet another index which, for what we said above, will refer to another Hilbert space $\mathcal{H_3}$; the system is, this time, a 5 level system (e.g. a boson truncated to the levels 0:4)

In [125]:
k = Index(5;tags="Boson, n=0:4")

(dim=5|id=5|"Boson,n=0:4")

We define the (truncated) bosonic operators

In [145]:
Diagonal([1,2,3])

3×3 Diagonal{Int64, Vector{Int64}}:
 1  ⋅  ⋅
 ⋅  2  ⋅
 ⋅  ⋅  3

In [168]:
aTruncMat = Tridiagonal(zeros(ComplexF64, 4), zeros(ComplexF64, 5), sqrt.(1:4))
aDagTruncMat = aTruncMat' #or adjoint(aTruncMat)
nMat = Diagonal(0:4)


5×5 Diagonal{Int64, UnitRange{Int64}}:
 0  ⋅  ⋅  ⋅  ⋅
 ⋅  1  ⋅  ⋅  ⋅
 ⋅  ⋅  2  ⋅  ⋅
 ⋅  ⋅  ⋅  3  ⋅
 ⋅  ⋅  ⋅  ⋅  4

In [169]:
aDagTruncMat * aTruncMat

5×5 Matrix{ComplexF64}:
 0.0+0.0im  0.0+0.0im  0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im  1.0+0.0im  0.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im  0.0+0.0im  2.0+0.0im  0.0+0.0im  0.0+0.0im
 0.0+0.0im  0.0+0.0im  0.0+0.0im  3.0+0.0im  0.0+0.0im
 0.0+0.0im  0.0+0.0im  0.0+0.0im  0.0+0.0im  4.0+0.0im

In [170]:
Diagonal(ComplexF64.(ones(5)))

5×5 Diagonal{ComplexF64, Vector{ComplexF64}}:
 1.0+0.0im      ⋅          ⋅          ⋅          ⋅    
     ⋅      1.0+0.0im      ⋅          ⋅          ⋅    
     ⋅          ⋅      1.0+0.0im      ⋅          ⋅    
     ⋅          ⋅          ⋅      1.0+0.0im      ⋅    
     ⋅          ⋅          ⋅          ⋅      1.0+0.0im

In [172]:
bosMat = [aTruncMat, aDagTruncMat, nMat,Diagonal(ComplexF64.(ones(5)))]
dOp3,aDagOp3,nOp3, id3 = tuple([ITensor(ComplexF64, mat, k', k) for mat in bosMat]...)

(ITensor ord=2
Dim 1: (dim=5|id=5|"Boson,n=0:4")'
Dim 2: (dim=5|id=5|"Boson,n=0:4")
NDTensors.Dense{ComplexF64, Base.ReshapedArray{ComplexF64, 1, Tridiagonal{ComplexF64, Vector{ComplexF64}}, Tuple{Base.MultiplicativeInverses.SignedMultiplicativeInverse{Int64}}}}
 5×5
 0.0 + 0.0im  1.0 + 0.0im                 0.0 + 0.0im  …  0.0 + 0.0im
 0.0 + 0.0im  0.0 + 0.0im  1.4142135623730951 + 0.0im     0.0 + 0.0im
 0.0 + 0.0im  0.0 + 0.0im                 0.0 + 0.0im     0.0 + 0.0im
 0.0 + 0.0im  0.0 + 0.0im                 0.0 + 0.0im     2.0 + 0.0im
 0.0 + 0.0im  0.0 + 0.0im                 0.0 + 0.0im     0.0 + 0.0im
, ITensor ord=2
Dim 1: (dim=5|id=5|"Boson,n=0:4")'
Dim 2: (dim=5|id=5|"Boson,n=0:4")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 5×5
 0.0 - 0.0im                 0.0 - 0.0im  …  0.0 - 0.0im  0.0 - 0.0im
 1.0 - 0.0im                 0.0 - 0.0im     0.0 - 0.0im  0.0 - 0.0im
 0.0 - 0.0im  1.4142135623730951 - 0.0im     0.0 - 0.0im  0.0 - 0.0im
 0.0 - 0.0im                 0.0 -

We now have:
- the indices for all Hilbert spaces: `i`, `j` and `k`
- the operators defined in the corresponding Hilbert spaces

In order to define a state in joined Hilbert space we need a rank-$3$ tensor:

In [ ]:
sysState = ITensor(ComplexF64, i,j,k)


ITensor ord=3 (dim=2|id=832|"Spin1/2") (dim=2|id=75|"Spin1/2") (dim=5|id=5|"Boson,n=0:4")
NDTensors.EmptyStorage{ComplexF64, NDTensors.Dense{ComplexF64, Vector{ComplexF64}}}

We can initialize each system into an arbitrary initial state

In [ ]:
#State |+>_1
vector(ψ1)

2-element Vector{ComplexF64}:
 0.7071067811865475 + 0.0im
 0.7071067811865475 + 0.0im

In [ ]:
#State |->_2
vector(ψ2)

2-element Vector{ComplexF64}:
  0.7071067811865475 + 0.0im
 -0.7071067811865475 + 0.0im

In [ ]:
#Oscillator state |0>_3 = (1,0,0,0,0),  i.e. vacuum state
ψ3 = ITensor(ComplexF64, ComplexF64.([1,0,0,0,0]), k) 

ITensor ord=1 (dim=5|id=5|"Boson,n=0:4")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

In [182]:
for a in 1:2, b in 1:2, c in 1:5
    sysState[i=>a, j=>b, k=>c] = ψ1[i=>a] * ψ2[j=>b] * ψ3[k=>c]
end

Norm of the resulting (product) state

In [192]:
scalar(sysState*dag(sysState))

0.9999999999999996 + 0.0im

For example,  if we want the expectation value of $\sigma^x_1 \otimes \mathbb{1}_2 \otimes \mathbb{1}_3$

In [202]:
inner(sysState',id3*id2*σx1,sysState)

0.9999999999999996 + 0.0im

or

In [ ]:
 scalar(noprime(σx1 * sysState)*dag(sysState))

To compute $\bra{\psi} \mathbb{1}_1 \otimes \sigma^x_2   \otimes \mathbb{1}_3 \ket{\psi}$

In [204]:
inner(sysState',id3*id1*σx2,sysState)

-0.9999999999999996 + 0.0im

If we want the average occupation number of the oscillator:

In [205]:
 scalar(noprime(nOp3 * sysState)*dag(sysState))

0.0 + 0.0im

### Partial Trace

We can obtain the density matrix of the composite system as:

In [211]:
ρsys = sysState * prime(dag(sysState))


ITensor ord=6 (dim=2|id=832|"Spin1/2") (dim=2|id=75|"Spin1/2") (dim=5|id=5|"Boson,n=0:4") (dim=2|id=832|"Spin1/2")' (dim=2|id=75|"Spin1/2")' (dim=5|id=5|"Boson,n=0:4")'
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

We can obtain the trace of the density matrix as

In [215]:
scalar(ρsys* delta(i, i')*delta(j, j')*delta(k, k'))

0.9999999999999996 + 0.0im

If we want the partial trace over, for example, the bosonic degree of freedom:

In [220]:
partialRes = ρsys * delta(k, k')

ITensor ord=4 (dim=2|id=832|"Spin1/2") (dim=2|id=75|"Spin1/2") (dim=2|id=832|"Spin1/2")' (dim=2|id=75|"Spin1/2")'
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

The resulting density matrix is equal to $\rho_1 \otimes \rho_2$, since the state is factorized.

In [221]:
sysStateReduced = ITensor(ComplexF64, i, j)
for a in 1:2, b in 1:2
    sysStateReduced[i=>a, j=>b] = ψ1[i=>a] * ψ2[j=>b]
end

In [219]:
ρRed = sysStateReduced * prime(dag(sysStateReduced))

ITensor ord=4 (dim=2|id=832|"Spin1/2") (dim=2|id=75|"Spin1/2") (dim=2|id=832|"Spin1/2")' (dim=2|id=75|"Spin1/2")'
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

In order to verify the identity we turn the density matrices $\rho$`Red` and `partialRes` into matrices. To this end it is expedient to use `combiner` objects: combiners, as their name suggests, combine different indices into one single index.

In [224]:
c = combiner(i,j)
cprime = combiner(prime(i), prime(j))

ITensor ord=3 (dim=4|id=354|"CMB,Link") (dim=2|id=832|"Spin1/2")' (dim=2|id=75|"Spin1/2")'
NDTensors.Combiner

In [227]:
ρRedComb =c * ρRed * cprime 
partialResComb = c * partialRes * cprime

ITensor ord=2 (dim=4|id=825|"CMB,Link") (dim=4|id=354|"CMB,Link")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

In [230]:
matrix(ρRedComb) == matrix(partialResComb)

true

### Example

Suppose now that we operate on the state before taking the paratial trace:

$\rho' = \frac{\mathbb{1} +\sigma_2^z}{2} (a_3 + a_3^\dag)\; \rho \; \frac{\mathbb{1} +\sigma_2^z}{2} (a_3 + a_3^\dag)$

This operation is well understood: the oscillator is shifted only if the spin 2 has non-vanishing amplitude in the state $\ket{\uparrow}$.

The implementation via ITensor constructs is a little bit involved. As per definition, the state tensor $\rho$`sys` is:

$\rho_S = \sum_{i,j,k, i',j',k'} \rho_{i,j,k}^{i',j',k'} \ket{i,j,j} \bra{i',j',k'}$

In [238]:
ρsys

ITensor ord=6 (dim=2|id=832|"Spin1/2") (dim=2|id=75|"Spin1/2") (dim=5|id=5|"Boson,n=0:4") (dim=2|id=832|"Spin1/2")' (dim=2|id=75|"Spin1/2")' (dim=5|id=5|"Boson,n=0:4")'
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

The operators are: $O = \sum_{i,i',j,j',k,k'} O_{i',j',k'}^{i,j,k} \ket{i',j',k'}\bra{i,j,k}$

We must therefore pay attention: if we contracted $\rho_S$ with $O$ we would get a rank-$0$ tensor, since __all__ of the indices would be contracted. What we want is something different. Even worse, we need to act on both the left side and the right side of $\rho_S$. 

In [240]:
ρsys*σz1*id2*id3

ITensor ord=0
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

We achieve the desired result by: delta tensors!!!!

In [248]:
poldo = prime(σz1*id2*id3,i',j',k')*ρsys

ITensor ord=6 (dim=2|id=832|"Spin1/2")'' (dim=2|id=75|"Spin1/2")'' (dim=5|id=5|"Boson,n=0:4")'' (dim=2|id=832|"Spin1/2")' (dim=2|id=75|"Spin1/2")' (dim=5|id=5|"Boson,n=0:4")'
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

In [250]:
pippo = poldo * σz1*id2*id3

ITensor ord=6 (dim=2|id=832|"Spin1/2")'' (dim=2|id=75|"Spin1/2")'' (dim=5|id=5|"Boson,n=0:4")'' (dim=2|id=832|"Spin1/2") (dim=2|id=75|"Spin1/2") (dim=5|id=5|"Boson,n=0:4")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

In [ ]:
set(poldo,)

ITensor ord=6 (dim=2|id=832|"Spin1/2") (dim=2|id=75|"Spin1/2") (dim=5|id=5|"Boson,n=0:4") (dim=2|id=832|"Spin1/2") (dim=2|id=75|"Spin1/2") (dim=5|id=5|"Boson,n=0:4")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

In [229]:
matrix(partialResComb)

4×4 Matrix{ComplexF64}:
  0.25+0.0im   0.25+0.0im  -0.25+0.0im  -0.25+0.0im
  0.25+0.0im   0.25+0.0im  -0.25+0.0im  -0.25+0.0im
 -0.25+0.0im  -0.25+0.0im   0.25+0.0im   0.25+0.0im
 -0.25+0.0im  -0.25+0.0im   0.25+0.0im   0.25+0.0im

In [235]:
inds(ρsys)

((dim=2|id=832|"Spin1/2"), (dim=2|id=75|"Spin1/2"), (dim=5|id=5|"Boson,n=0:4"), (dim=2|id=832|"Spin1/2")', (dim=2|id=75|"Spin1/2")', (dim=5|id=5|"Boson,n=0:4")')

In [236]:
evpsi = noprime(σz1*σy2*sysState)

ITensor ord=3 (dim=2|id=832|"Spin1/2") (dim=2|id=75|"Spin1/2") (dim=5|id=5|"Boson,n=0:4")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}

In [237]:
evpsi*prime(dag(evpsi))

ITensor ord=6 (dim=2|id=832|"Spin1/2") (dim=2|id=75|"Spin1/2") (dim=5|id=5|"Boson,n=0:4") (dim=2|id=832|"Spin1/2")' (dim=2|id=75|"Spin1/2")' (dim=5|id=5|"Boson,n=0:4")'
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}